# Tiền xử lý bước 1

## Bước này tạo file:

- df_inter.label.parquet.

File này chia dữ liệu theo x_label có 3 giá trị: 0, 1, 2 tương ứng với train, val, test.


# Train/Validation/Test data splitting

- Based on generated interactions, perform data splitting


In [1]:
import os
import pandas as pd

In [2]:
PATH = "./data/2014"

## Load interactions


In [3]:
df = pd.read_parquet(os.path.join(PATH, "df_inter.parquet"))

In [4]:
print(f"shape: {df.shape}")
df[:4]

shape: (160792, 6)


,userID,itemID,rating,timestamp,reviewerID,asin
0,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X
1,1,0,5.0,1372464000,A19K65VY14D13R,097293751X
2,2,0,5.0,1395187200,A2LL1TGG90977E,097293751X
3,3,0,5.0,1376697600,A5G19RYX8599E,097293751X


Đoạn code này thực hiện hai thao tác xử lý dữ liệu có vẻ trái ngược nhau nhưng lại rất phổ biến trong quá trình chuẩn bị dữ liệu cho hệ thống gợi ý.


In [5]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.sort_values(by=["userID", "timestamp"], inplace=True)

df[:10]

,userID,itemID,rating,timestamp,reviewerID,asin
16269,0,3870,4.0,1357689600,A1HK2FQW6KXQB2,B004071ZOY
123420,0,1922,5.0,1363996800,A1HK2FQW6KXQB2,B001F8TLLU
19680,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X
40627,0,1587,5.0,1373932800,A1HK2FQW6KXQB2,B0013FGWD0
50552,0,1879,5.0,1373932800,A1HK2FQW6KXQB2,B001DKHPD6
35674,1,5403,5.0,1367452800,A19K65VY14D13R,B006OK476S
85844,1,1434,5.0,1372377600,A19K65VY14D13R,B000YDIGCC
38492,1,2828,5.0,1372464000,A19K65VY14D13R,B002QBDMDI
92863,1,0,5.0,1372464000,A19K65VY14D13R,097293751X
99692,1,1177,5.0,1372464000,A19K65VY14D13R,B000Q7FY26


In [6]:
# 1. Khai báo tên cột
uid_field, iid_field = "userID", "itemID"

# 2. Nhóm dữ liệu (Groupby)
uid_freq = df.groupby(uid_field)[iid_field]
u_i_dict = {}
for u, u_ls in uid_freq:
    u_i_dict[u] = list(u_ls)
list(u_i_dict.items())[:3]

[(0, [3870, 1922, 0, 1587, 1879]),
 (1, [5403, 1434, 2828, 0, 1177, 5176]),
 (2, [6578, 4011, 6735, 2846, 0])]

In [7]:
list(u_i_dict.keys())[:3]

[0, 1, 2]

In [8]:
new_label = []
u_ids_sorted = sorted(u_i_dict.keys())
for u in u_ids_sorted:
    items = u_i_dict[u]
    # get num interact
    n_items = len(items)
    if n_items < 10:
        # take 1 for test 1 for val and rest for train
        tmp_ls = [0] * (n_items - 2) + [1] + [2]
    else:
        # split 80% train, 10% val, 10% test
        val_test_len = int(n_items * 0.2)
        train_len = n_items - val_test_len
        val_len = val_test_len // 2
        test_len = val_test_len - val_len
        tmp_ls = [0] * train_len + [1] * val_len + [2] * test_len
    new_label.extend(tmp_ls)

new_label[:10]

[0, 0, 0, 1, 2, 0, 0, 0, 0, 1]

In [10]:
df["x_label"] = new_label
df[:20]

,userID,itemID,rating,timestamp,reviewerID,asin,x_label
16269,0,3870,4.0,1357689600,A1HK2FQW6KXQB2,B004071ZOY,0
123420,0,1922,5.0,1363996800,A1HK2FQW6KXQB2,B001F8TLLU,0
19680,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X,0
40627,0,1587,5.0,1373932800,A1HK2FQW6KXQB2,B0013FGWD0,1
50552,0,1879,5.0,1373932800,A1HK2FQW6KXQB2,B001DKHPD6,2
35674,1,5403,5.0,1367452800,A19K65VY14D13R,B006OK476S,0
85844,1,1434,5.0,1372377600,A19K65VY14D13R,B000YDIGCC,0
38492,1,2828,5.0,1372464000,A19K65VY14D13R,B002QBDMDI,0
92863,1,0,5.0,1372464000,A19K65VY14D13R,097293751X,0
99692,1,1177,5.0,1372464000,A19K65VY14D13R,B000Q7FY26,1


In [11]:
df.to_parquet(os.path.join(PATH, "df_inter.label.parquet"), index=False)

## Reload


In [12]:
indexed_df = pd.read_parquet(os.path.join(PATH, "df_inter.label.parquet"))
print(f"shape: {indexed_df.shape}")
indexed_df[:20]

shape: (160792, 7)


,userID,itemID,rating,timestamp,reviewerID,asin,x_label
0,0,3870,4.0,1357689600,A1HK2FQW6KXQB2,B004071ZOY,0
1,0,1922,5.0,1363996800,A1HK2FQW6KXQB2,B001F8TLLU,0
2,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X,0
3,0,1587,5.0,1373932800,A1HK2FQW6KXQB2,B0013FGWD0,1
4,0,1879,5.0,1373932800,A1HK2FQW6KXQB2,B001DKHPD6,2
5,1,5403,5.0,1367452800,A19K65VY14D13R,B006OK476S,0
6,1,1434,5.0,1372377600,A19K65VY14D13R,B000YDIGCC,0
7,1,2828,5.0,1372464000,A19K65VY14D13R,B002QBDMDI,0
8,1,0,5.0,1372464000,A19K65VY14D13R,097293751X,0
9,1,1177,5.0,1372464000,A19K65VY14D13R,B000Q7FY26,1


In [13]:
train_df = indexed_df[["userID", "itemID", "rating", "timestamp", "x_label"]].copy()
train_df

,userID,itemID,rating,timestamp,x_label
0,0,3870,4.0,1357689600,0
1,0,1922,5.0,1363996800,0
2,0,0,5.0,1373932800,0
3,0,1587,5.0,1373932800,1
4,0,1879,5.0,1373932800,2
...,...,...,...,...,...
160787,19444,6994,5.0,1393632000,0
160788,19444,7022,5.0,1393804800,0
160789,19444,6959,5.0,1394064000,0
160790,19444,7023,4.0,1394668800,1


In [14]:
# Lưu lại file .inter để train, tùy dataset mà tên bỏ vào thư mục data sẽ khác
# ví dụ data/baby/baby.inter, data/beauty/beauty.inter, data/clothing/clothing.inter
train_df.to_csv(os.path.join(PATH, "df_inter.label.inter"), sep="\t")

In [25]:
u_id_str, i_id_str = "userID", "itemID"
u_uni = indexed_df[u_id_str].unique()
c_uni = indexed_df[i_id_str].unique()

print(f"# of unique learners: {len(u_uni)}")
print(f"# of unique courses: {len(c_uni)}")

print("min/max of unique learners: {0}/{1}".format(min(u_uni), max(u_uni)))
print("min/max of unique courses: {0}/{1}".format(min(c_uni), max(c_uni)))

# of unique learners: 19372
# of unique courses: 7025
min/max of unique learners: 0/19371
min/max of unique courses: 0/7024
